# Portfolio 3 — AquaSense+ transactions and concurrency: run and test

## 1. Imports

In [1]:
import os, sys, platform, getpass, subprocess
import pandas as pd
os.makedirs("evidence", exist_ok=True)
print("Files here:", sorted(os.listdir()))

Files here: ['.ipynb_checkpoints', '01_transaction_procedures.sql', '02_commit_rollback_savepoint.sql', '03_concurrency_demos.py', 'README.md', 'evidence', 'portfolio3_run.ipynb']


## 2. Connection settings
A box appears asking for your PostgreSQL password, so it is never saved in the notebook.

In [2]:
if platform.system() == "Windows":
    os.environ["PATH"] += r";C:\Program Files\PostgreSQL\16\bin"
os.environ["PGUSER"] = "postgres"
os.environ["PGPASSWORD"] = getpass.getpass("PostgreSQL password: ")
os.environ["PGHOST"] = "localhost"
os.environ["PGDATABASE"] = "aquasense"
os.environ["PGCLIENTENCODING"] = "UTF8"
os.environ["PYTHONUTF8"] = "1"          # lets the demo script print dashes and arrows on Windows

!psql -c "SELECT count(*) AS active_batches FROM aqua.stock_batch WHERE status = 'active'"

PostgreSQL password:  ········


 active_batches 
----------------
            745
(1 row)



## 3. Install the three transactional procedures
- `record_feeding` — inserts the feeding log **and** reduces feed stock in one transaction, with a row lock
- `sell_fish` — blocks selling more fish than was harvested
- `transfer_feed` — moves feed between two farms, locking rows in a fixed order to avoid deadlocks

In [3]:
!psql -q -v ON_ERROR_STOP=1 -f 01_transaction_procedures.sql
!psql -c "SELECT p.proname AS name, CASE p.prokind WHEN 'p' THEN 'procedure' ELSE 'function' END AS kind, pg_get_function_arguments(p.oid) AS arguments FROM pg_proc p JOIN pg_namespace n ON n.oid = p.pronamespace WHERE n.nspname = 'aqua' AND p.proname IN ('record_feeding','sell_fish','transfer_feed')"

      name      |   kind    |                                 arguments                                  
----------------+-----------+----------------------------------------------------------------------------
 record_feeding | procedure | IN p_batch integer, IN p_feed integer, IN p_qty numeric, IN p_user integer
 sell_fish      | function  | p_harvest integer, p_trader integer, p_qty numeric, p_price numeric
 transfer_feed  | procedure | IN p_from integer, IN p_to integer, IN p_feed integer, IN p_qty numeric
(3 rows)



## 4. COMMIT, ROLLBACK and SAVEPOINT
Six scenarios. You should see **4 ERROR lines**, and each one is a rule doing its job:
1. Feeding 500 kg with only 70 kg in store → the whole transaction rolls back
2. A mistyped pH of 19.5 during offline sync → only that record is undone with `ROLLBACK TO SAVEPOINT`
3. Selling 99,999 kg from a harvest → blocked
4. Transferring more feed than a farm holds → blocked, nothing moves

The output is saved to `evidence/commit_rollback_savepoint_output.txt`.

In [4]:
!psql -f 02_commit_rollback_savepoint.sql > evidence/commit_rollback_savepoint_output.txt 2>&1
out = open("evidence/commit_rollback_savepoint_output.txt", encoding="utf-8", errors="replace").read()
print(out)
print("ERROR lines:", out.count("ERROR:"))

SET
INSERT 0 1
Demo batch: 2717

=== 1. COMMIT: record 30 kg of feed — log inserted AND stock reduced together ===
BEGIN
CALL
COMMIT
 stock_after_commit 
--------------------
              70.00
(1 row)

 new_logs 
----------
        1
(1 row)


=== 2. AUTOMATIC ROLLBACK: try to feed 500 kg when only 70 kg is in store ===
BEGIN
psql:02_commit_rollback_savepoint.sql:28: ERROR:  Not enough feed: 70.00 kg in store, 500 kg requested
CONTEXT:  PL/pgSQL function record_feeding(integer,integer,numeric,integer) line 18 at RAISE
ROLLBACK
 stock_unchanged 
-----------------
           70.00
(1 row)

 new_logs_still 
----------------
              1
(1 row)


=== 3. EXPLICIT ROLLBACK: attendant records 20 kg, then cancels (wrong pond) ===
BEGIN
CALL
 stock_inside_txn 
------------------
            50.00
(1 row)

ROLLBACK
 stock_after_rollback 
----------------------
                70.00
(1 row)


=== 4. SAVEPOINT: offline sync of 4 queued records — one bad record must not lose the rest ===
BEGI

## 5. Concurrency demos
Two real database sessions, A and B, race each other on the same rows. The timestamps show the interleaving.

| Demo | What you should see |
|---|---|
| D1 | Lost update: final stock **80 kg instead of 50** |
| D2 | `FOR UPDATE`: B **waits about 1 s**, result 50 kg |
| D3 | REPEATABLE READ: B gets **40001**, retries, result 50 kg |
| D4 | READ COMMITTED reads 100 then 250; REPEATABLE READ reads 100 both times |
| D5 | REPEATABLE READ oversells **1,200 kg of 1,000**; SERIALIZABLE aborts one (40001); `sell_fish` blocks it (AQ003) |
| D6 | **Deadlock** (40P01), then lock ordering makes B queue instead, total stays 200 kg |

The output is saved to `evidence/concurrency_demo_output.txt`.

In [5]:
run = subprocess.run([sys.executable, "03_concurrency_demos.py"], capture_output=True, text=True, encoding="utf-8", errors="replace")
print(run.stdout)
print(run.stderr)
open("evidence/concurrency_demo_output.txt", "w", encoding="utf-8").write(run.stdout)


D1  LOST UPDATE — READ COMMITTED, app reads stock then writes the new value
    Store has 100 kg. Attendant A records 30 kg, attendant B records 20 kg. Correct result: 50 kg.
[  0.29s] A       SELECT qty_kg FROM feed_inventory WHERE farm_id = 1 AND feed_id = 4                        -> [(Decimal('100.00'),)]
[  0.30s] B       SELECT qty_kg FROM feed_inventory WHERE farm_id = 1 AND feed_id = 4                        -> [(Decimal('100.00'),)]
[  0.30s] A       UPDATE feed_inventory SET qty_kg = %s WHERE farm_id = 1 AND feed_id = 4                    -> UPDATE 1
[  0.30s] A       COMMIT
[  0.30s] B       UPDATE feed_inventory SET qty_kg = %s WHERE farm_id = 1 AND feed_id = 4                    -> UPDATE 1
[  0.30s] B       COMMIT
[  0.37s] CHECK   final stock = 80.00 kg  (expected 50) -> A's 30 kg deduction was LOST

D2  FIX 1 — pessimistic locking: SELECT ... FOR UPDATE
    B's read blocks until A commits, then B sees A's result.
[  0.60s] A       SELECT qty_kg FROM feed_inventory WHERE

10162

## 6. The key results at a glance
Every `CHECK` line is the outcome of one demo; `waited` lines are lock waits; `ERROR` lines are the database stopping an anomaly.

In [6]:
for l in run.stdout.splitlines():
    if any(k in l for k in ("CHECK", "waited", "ERROR")):
        print(l[:170])

[  0.37s] CHECK   final stock = 80.00 kg  (expected 50) -> A's 30 kg deduction was LOST
[  1.80s] B       (waited 1.19s for A's row lock)
[  1.91s] CHECK   final stock = 50.00 kg  (expected 50) -> correct
[  2.15s] B       ERROR 40001 ERROR:  could not serialize access due to concurrent update
[  2.30s] CHECK   final stock = 50.00 kg  (expected 50) -> correct after one retry
[  2.60s] CHECK   A read 100.00 then 250.00 -> CHANGED inside one transaction (non-repeatable read)
[  2.87s] CHECK   A read 100.00 then 100.00 -> STABLE (same snapshot for the whole transaction)
[  3.43s] CHECK   total sold = 1200.00 kg of 1000 kg -> OVERSOLD (anomaly allowed)
[  3.75s] B       ERROR 40001 ERROR:  could not serialize access due to read/write dependencies among transactions
[  3.85s] CHECK   total sold = 600.00 kg of 1000 kg -> rule held
[  5.18s] B       ERROR AQ003 ERROR:  Oversell blocked: 1000.00 kg harvested, 600.00 kg already sold, 600 kg requested
[  5.32s] CHECK   total sold = 600.00 kg of 

## 7. Numbers to copy into the README


In [7]:
import re
out2 = open("evidence/commit_rollback_savepoint_output.txt", encoding="utf-8", errors="replace").read()
waits = re.findall(r"waited ([0-9.]+)s", run.stdout)
victim = re.search(r"outcome: \{'A': '([^']+)', 'B': '([^']+)'\}", run.stdout)
row_id = re.search(r"Failing row contains \((\d+)", out2)
harvest = re.search(r"Oversell blocked: ([0-9.]+) kg harvested", out2)
pd.DataFrame([
    {"Item": "D2 wait for A's row lock (s)", "Your value": waits[0] if waits else "?"},
    {"Item": "D6b queue time (s)", "Your value": waits[1] if len(waits) > 1 else "?"},
    {"Item": "D6 deadlock: A was", "Your value": victim.group(1) if victim else "?"},
    {"Item": "D6 deadlock: B was", "Your value": victim.group(2) if victim else "?"},
    {"Item": "reading_id given to the rejected pH row", "Your value": row_id.group(1) if row_id else "?"},
    {"Item": "Harvest 1 weight (kg)", "Your value": harvest.group(1) if harvest else "?"},
])

,Item,Your value
0,D2 wait for A's row lock (s),1.19
1,D6b queue time (s),0.80
2,D6 deadlock: A was,aborted (deadlock victim)
3,D6 deadlock: B was,committed
4,reading_id given to the rejected pH row,1233614
5,Harvest 1 weight (kg),1319.06
